#Enterprise Fleet Analytics Pipeline: Focuses on the business outcome (analytics) and the domain (fleet/logistics).

![logistics](logistics_project.png)

Download the data from the below gdrive and upload into the catalog
https://drive.google.com/drive/folders/1J3AVJIPLP7CzT15yJIpSiWXshu1iLXKn?usp=drive_link

##**1. Data Munging** -

####1. Visibily/Manually opening the file and capture couple of data patterns (Manual Exploratory Data Analysis)

######Shipment id is having null and char in integer column
######Firstname and Lastname columns having null
######Additional column value for one records
######Data tye uniformity error 

####2. Programatically try to find couple of data patterns applying below EDA (File: logistics_source1)
1. Apply inferSchema and toDF to create a DF and analyse the actual data.
2. Analyse the schema, datatypes, columns etc.,
3. Analyse the duplicate records count and summary of the dataframe.

In [0]:
rawlogis1 = spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_source1",header = True,inferSchema = True).\
toDF("shipment_id","first_name","last_name","age","role")
#display(rawlogis1)
rawlogis2 = spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_source2",header = True,inferSchema = True).\
toDF("shipment_id","first_name","last_name","age","role","hub_location","vehicle_type")
#display(rawlogis2)
print(rawlogis1.columns)
print(rawlogis2.columns)
print(rawlogis1.dtypes)
print(rawlogis2.dtypes)
print(rawlogis1.schema)
print(rawlogis2.schema)
for i in rawlogis1.dtypes:
  if i[1] == 'string':
      print(i[0])
for i in rawlogis2.dtypes:
    if i[1] == 'string':
        print(i[0])



###a. Passive Data Munging -  (File: logistics_source1  and logistics_source2)
Without modifying the data, identify:<br>
Shipment IDs that appear in both master_v1 and master_v2<br>
Records where:<br>
1. shipment_id is non-numeric
2. age is not an integer<br>

Count rows having:
3. fewer columns than expected
4. more columns than expected

In [0]:
from pyspark.sql.functions import col
test = rawlogis1.where(col("shipment_id").rlike("[A-Za-z]"))
test1 = rawlogis2.where(col("age").rlike("[^0-9]"))
display(test1)
display(test)
print(len(rawlogis1.columns))

In [0]:
#Create a Spark Session Object

###**b. Active Data Munging** File: logistics_source1 and logistics_source2

#####1.Combining Data + Schema Merging (Structuring)
1. Read both files without enforcing schema
2. Align them into a single canonical schema: shipment_id,
first_name,
last_name,
age,
role,
hub_location,
vehicle_type,
data_source
3. Add data_source column with values as: system1, system2 in the respective dataframes

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,ShortType,LongType
from pyspark.sql.functions import col,lit

#1.
rawdf1 = spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_source1",header = True,inferSchema = False)
rawdf2 = spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_source2",header = True,inferSchema = False)
#display(rawdf1.limit(5))
#display(rawdf2.limit(5))

#2.
struct_type = StructType([
  StructField("shipment_id", IntegerType(), True),
  StructField("first_name", StringType(), True),
  StructField("last_name", StringType(), True),
  StructField("age", StringType(), True),
  StructField("role", StringType(), True),
  StructField("hub_location", StringType(), True),
  StructField("vehicle_type", StringType(), True),
  StructField("data_source", StringType(), True)])

rawdf1 = spark.read.schema(struct_type).csv("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_source1",header = True,inferSchema = False).toDF("shipment_id","first_name","last_name","age","role","hub_location","vehicle_type","data_source").withColumn("data_source",lit("system1"))
#display(rawdf1)
rawdf2 = spark.read.schema(struct_type).csv("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_source2",header = True,inferSchema = False).toDF("shipment_id","first_name","last_name","age","role","hub_location","vehicle_type","data_source").withColumn("data_source",lit("system2"))
#display(rawdf2)

rawdf3 = rawdf1.unionByName(rawdf2,allowMissingColumns=True)
display(rawdf3)




#####2. Cleansing, Scrubbing: 
Cleansing (removal of unwanted datasets)<br>
1. Mandatory Column Check - Drop any record where any of the following columns is NULL:shipment_id, role<br>
2. Name Completeness Rule - Drop records where both of the following columns are NULL: first_name, last_name<br>
3. Join Readiness Rule - Drop records where the join key is null: shipment_id<br>

Scrubbing (convert raw to tidy)<br>
4. Age Defaulting Rule - Fill NULL values in the age column with: -1<br>
5. Vehicle Type Default Rule - Fill NULL values in the vehicle_type column with: UNKNOWN<br>
6. Invalid Age Replacement - Replace the following values in age:
"ten" to -1
"" to -1<br>
7. Vehicle Type Normalization - Replace inconsistent vehicle types: 
truck to LMV
bike to TwoWheeler

In [0]:
#1.Mandatory Column Check - Drop any record where any of the following columns is NULL:shipment_id, role
rawdf4 = rawdf3.where("shipment_id is null or role is null")
rawdf5 = rawdf3.where("first_name is null and last_name is null")
display(rawdf4.count())
display(" ")
display(rawdf5.count())
display(" ")
rawdf4  = rawdf3.na.drop(how = "any",subset = ["shipment_id","role"])
rawdf5 = rawdf4.na.drop(how = "all", subset = ["first_name","last_name"])
#display(rawdf5.count())
#display(rawdf5)
#2. Name Completeness Rule - Drop records where both of the following columns are NULL: first_name, last_name
rawdf5 = rawdf4.na.drop(how = "all", subset = ["first_name","last_name"])
display(rawdf5.count())
display(rawdf5)
##3.Join Readiness Rule - Drop records where the join key is null: shipment_id
#4.Age Defaulting Rule - Fill NULL values in the age column with: -1
rawdf6 = rawdf5.na.fill(-1,subset=["age"])
#5. Vehicle Type Default Rule - Fill NULL values in the vehicle_type column with: UNKNOWN
rawdf6 = rawdf6.na.fill("UNKNOWN",subset=["vehicle_type"])
display(rawdf6)
#6. Invalid Age Replacement - Replace the following values in age: "ten" to -1 "" to -1
#7. Vehicle Type Normalization - Replace inconsistent vehicle types: truck to LMV bike to TwoWheeler
agedict = {'ten':'-1','one':'1','two':'2','three':'3','four':'4','five':'5','six':'6','seven':'7','eight':'8','nine':'9','':'-1'}
dict2 = {'Truck':'LMV','Bike':'TwoWheeler'}
dict3 = {'null':'-1'}
rawdf7 = rawdf6.na.replace(agedict,subset=["age"])
rawdf8 = rawdf7.na.replace(dict2,subset=["vehicle_type"])
rawdf9 = rawdf8.withColumn("age",col("age").cast(IntegerType()))
rawdf10 = rawdf9.na.fill(-1,subset=["age"])
display(rawdf10)


####3. Standardization, De-Duplication and Replacement / Deletion of Data to make it in a usable format

Detail Dataframe creation <br>
1. Read Data from logistics_shipment_detail.json
2. As this data is a clean json data, it doesn't require any cleansing or scrubbing.

Standardizations:<br>

1. Add a column<br> 
Source File: logistics_shipment_detail_3000.json<br>: domain as 'Logistics',  current timestamp 'ingestion_timestamp' and 'False' as 'is_expedited'
2. Column Uniformity: 
role - Convert to lowercase<br>
Source File: logistics_source1 & logistics_source2<br>
vehicle_type - Convert values to UPPERCASE<br>
Source Files: logistics_shipment_detail_3000.json (and the merged master files)
hub_location - Convert values to initcap case<br>
3. Format Standardization:<br>
Source Files: logistics_shipment_detail_3000.json<br>
Convert shipment_date to yyyy-MM-dd<br>
Ensure shipment_cost has 2 decimal precision<br>
4. Data Type Standardization<br>
Standardizing column data types to fix schema drift and enable mathematical operations.<br>
Source File: logistics_source1 & logistics_source2 <br>
age: Cast String to Integer<br>
Source File: logistics_shipment_detail_3000.json<br>
shipment_weight_kg: Cast to Double<br>
Source File: logistics_shipment_detail_3000.json<br>
is_expedited: Cast to Boolean<br>
5. Naming Standardization <br>
Source File: logistics_source1 & logistics_source2<br>
Rename: first_name to staff_first_name<br>
Rename: last_name to staff_last_name<br>
Rename: hub_location to origin_hub_city<br>
6. Reordering columns logically in a better standard format:<br>
Source File: All 3 files<br>
shipment_id (Identifier), staff_first_name (Dimension)staff_last_name (Dimension), role (Dimension), origin_hub_city (Location), shipment_cost (Metric), ingestion_timestamp (Audit)

In [0]:

struct_json = StructType([
  StructField("shipment_id", IntegerType(), True),
  StructField("order_id", StringType(), True),
  StructField("source_city", StringType(), True),
  StructField("destination_city", StringType(), True),
  StructField("shipment_status", StringType(), True),
  StructField("cargo_type", StringType(), True),
  StructField("vehicle_type", StringType(), True),
  StructField("payment_mode", StringType(), True),
  StructField("shipment_weight_kg", StringType(), True),
  StructField("shipment_cost", StringType(), True),
  StructField("shipment_date", StringType(), True)])


jsondf = spark.read \
    .option("mode", "PERMISSIVE") \
    .schema(struct_json)\
    .option("multiLine", "true")\
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .json("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_shipment_detail_3000.json")
display(jsondf.limit(5))

In [0]:
from pyspark.sql.functions import lit,lower,upper,col,initcap,current_timestamp,now,current_date,to_date,round,initcap
#1.Add a column
#Source File: logistics_shipment_detail_3000.json
#: domain as 'Logistics', current timestamp 'ingestion_timestamp' and 'False' as 'is_expedited'
jsondf1 = jsondf.withColumn("domain",lit("Logistics")).withColumn("ingestion_timestamp",current_date()).withColumn("is_expedited",lit("False"))
display(jsondf1.limit(2))  

#2Column Uniformity: role - Convert to lowercase
#Source File: logistics_source1 & logistics_source2
#vehicle_type - Convert values to UPPERCASE
##Source Files: logistics_shipment_detail_3000.json (and the merged master files) hub_location - Convert values to initcap case
rawdf11 = rawdf10.withColumn("role",lower(col("role"))).withColumn("vehicle_type",upper(col("vehicle_type"))).withColumn("hub_location",initcap(col("hub_location")))
display(rawdf11.limit(2))

#3.Format Standardization:
#Source Files: logistics_shipment_detail_3000.json
#Convert shipment_date to yyyy-MM-dd
#Ensure shipment_cost has 2 decimal precision
jsondf2 = jsondf1.withColumn("shipment_date",to_date(col("shipment_date"),"yy-MM-dd")).withColumn("shipment_cost",round(col("shipment_cost").cast("double"),2))
display(jsondf2.limit(2))




In [0]:
rjson = spark.read.json("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_shipment_detail_3000.json").cache()
display(rjson)

In [0]:
#4.Data Type Standardization
#Standardizing column data types to fix schema drift and enable mathematical operations.
#Source File: logistics_source1 & logistics_source2
#age: Cast String to Integer
#Source File: logistics_shipment_detail_3000.json
#shipment_weight_kg: Cast to Double
#Source File: logistics_shipment_detail_3000.json
#is_expedited: Cast to Boolean

rawdf12 = rawdf11.withColumn("age",col("age").cast("Integer"))
display(rawdf12.limit(2))

jsondf3 = jsondf2.withColumn("shipment_weight_kg",(round(col("shipment_weight_kg").cast("double"),2))).withColumn("is_expedited",initcap(col("is_expedited").cast("boolean")))
display(jsondf3.limit(5))

#5.Naming Standardization
#Source File: logistics_source1 & logistics_source2
#Rename: first_name to staff_first_name
#Rename: last_name to staff_last_name
#Rename: hub_location to origin_hub_city

rawdf11 = rawdf11.withColumnRenamed("first_name","staff_first_name").withColumnRenamed("last_name","staff_last_name").withColumnRenamed("hub_location","origin_hub_city")
display(rawdf11.limit(5))

#6.Reordering columns logically in a better standard format:
#Source File: All 3 files
#shipment_id (Identifier), staff_first_name (Dimension)staff_last_name (Dimension), role (Dimension), origin_hub_city (Location), shipment_cost (Metric), ingestion_timestamp (Audit)

combdf = rawdf11.select("shipment_id","staff_first_name","staff_last_name","role","origin_hub_city")
comb1df = jsondf3.select("shipment_id","shipment_cost","ingestion_timestamp")
combdf2 = combdf.unionByName(comb1df,allowMissingColumns=True)
combdf3 = combdf.join(comb1df,how = "left",on = "shipment_id")
display(combdf3)

Deduplication:
1. Apply Record Level De-Duplication
2. Apply Column Level De-Duplication (Primary Key Enforcement)

In [0]:

#rawdf13 = rawdf12.distinct().count()
rawdf13 = rawdf12.coalesce(1).dropDuplicates(subset = ["shipment_id"])
display(rawdf12)
display(rawdf13)



##2. Data Enrichment - Detailing of data
Makes your data rich and detailed <br>

###### Adding of Columns (Data Enrichment)
*Creating new derived attributes to enhance traceability and analytical capability.*

**1. Add Audit Timestamp (`load_dt`)**
Source File: DF of logistics_source1 and logistics_source2<br>
* **Scenario:** We need to track exactly when this record was ingested into our Data Lakehouse for auditing purposes.
* **Action:** Add a column `load_dt` using the function `current_timestamp()`.

**2. Create Full Name (`full_name`)**
Source File: DF of logistics_source1 and logistics_source2<br>
* **Scenario:** The reporting dashboard requires a single field for the driver's name instead of separate columns.
* **Action:** Create `full_name` by concatenating `first_name` and `last_name` with a space separator.
* **Result:** "Rajesh" + " " + "Kumar" -> **"Rajesh Kumar"**

**3. Define Route Segment (`route_segment`)**
Source File: DF of logistics_shipment_detail_3000.json<br>
* **Scenario:** The logistics team wants to analyze performance based on specific transport lanes (Source to Destination).
* **Action:** Combine `source_city` and `destination_city` with a hyphen.
* **Result:** "Chennai" + "-" + "Pune" -> **"Chennai-Pune"**

**4. Generate Vehicle Identifier (`vehicle_identifier`)**
Source File: DF of logistics_shipment_detail_3000.json<br>
* **Scenario:** We need a unique tracking code that immediately tells us the vehicle type and the shipment ID.
* **Action:** Combine `vehicle_type` and `shipment_id` to create a composite key.
* **Result:** "Truck" + "_" + "500001" -> **"Truck_500001"**

In [0]:
from pyspark.sql.functions import current_timestamp,date_format,concat,when
#1. Add Audit Timestamp (load_dt) Source File: DF of logistics_source1 and logistics_source2
#Scenario: We need to track exactly when this record was ingested into our Data Lakehouse for auditing purposes.
#Action: Add a column load_dt using the function current_timestamp().

#2. Create Full Name (full_name) Source File: DF of logistics_source1 and logistics_source2
#Scenario: The reporting dashboard requires a single field for the driver's name instead of separate columns.
#Action: Create full_name by concatenating first_name and last_name with a space separator.
#Result: "Rajesh" + " " + "Kumar" -> "Rajesh Kumar"

rawdf14 = rawdf13.withColumn("load_dt",date_format(current_timestamp(),"yyyy-MM-dd HH:mm:ss")).na.fill("UNKNOWN",subset = ["first_name","last_name"]).\
withColumn("fullname",
            when(col("first_name") != "UNKNOWN", concat(col("first_name"),lit(" "),col("last_name"))).otherwise(col("last_name"))).\
            withColumnsRenamed({"first_name":"staff_first_name","last_name":"staff_last_name"})
display(rawdf14.limit(5))


In [0]:
#3. Define Route Segment (route_segment) Source File: DF of logistics_shipment_detail_3000.json
#Scenario: The logistics team wants to analyze performance based on specific transport lanes (Source to Destination).
#Action: Combine source_city and destination_city with a hyphen.
#Result: "Chennai" + "-" + "Pune" -> "Chennai-Pune"

#4. Generate Vehicle Identifier (vehicle_identifier) Source File: DF of logistics_shipment_detail_3000.json
#Scenario: We need a unique tracking code that immediately tells us the vehicle type and the shipment ID.
#Action: Combine vehicle_type and shipment_id to create a composite key.
#Result: "Truck" + "_" + "500001" -> "Truck_500001"

jsondf4 = jsondf3.withColumn("route_segment",concat(col("source_city"),lit("-"),col("destination_city"))).\
                  withColumn("vehicle_identifier",concat(col("vehicle_type"),lit("_"),col("shipment_id")))

display(jsondf4)

###### Deriving of Columns (Time Intelligence)
*Extracting temporal features from dates to enable period-based analysis and reporting.*<br>
Source File: logistics_shipment_detail_3000.json<br>
**1. Derive Shipment Year (`shipment_year`)**
* **Scenario:** Management needs an annual performance report to compare growth year-over-year.
* **Action:** Extract the year component from `shipment_date`.
* **Result:** "2024-04-23" -> **2024**

**2. Derive Shipment Month (`shipment_month`)**
* **Scenario:** Analysts want to identify seasonal peaks (e.g., increased volume in December).
* **Action:** Extract the month component from `shipment_date`.
* **Result:** "2024-04-23" -> **4** (April)

**3. Flag Weekend Operations (`is_weekend`)**
* **Scenario:** The Operations team needs to track shipments handled during weekends to calculate overtime pay or analyze non-business day capacity.
* **Action:** Flag as **'True'** if the `shipment_date` falls on a Saturday or Sunday.

In [0]:
from pyspark.sql.functions import year,month,dayofweek,when,col,date_diff
#1.Derive Shipment Year (shipment_year)
#Scenario: Management needs an annual performance report to compare growth year-over-year.
#Action: Extract the year component from shipment_date.
#Result: "2024-04-23" -> 2024
#2. Derive Shipment Month (shipment_month)
#Scenario: Analysts want to identify seasonal peaks (e.g., increased volume in December).
#Action: Extract the month component from shipment_date.
#Result: "2024-04-23" -> 4 (April)

#3. Flag Weekend Operations (is_weekend)
#Scenario: The Operations team needs to track shipments handled during weekends to calculate overtime pay or analyze non-business day capacity.
#Action: Flag as 'True' if the shipment_date falls on a Saturday or Sunday.

jsondf5 = jsondf4.withColumn("shipment_year",year(col("shipment_date"))).withColumn("shipment_month",month(col("shipment_date"))).\
withColumn("is_weekend",when(dayofweek(col("shipment_date")).isin([1,7]),"True").otherwise("False"))#.\
#withColumn("is_weekday",when(dayofweek(col("shipment_date")).isin([2,3,4,5,6]),"True").otherwise("False"))    
display(jsondf5)



###### Enrichment/Business Logics (Calculated Fields)
*Deriving new metrics and financial indicators using mathematical and date-based operations.*<br>
Source File: logistics_shipment_detail_3000.json<br>

**1. Calculate Unit Cost (`cost_per_kg`)**
* **Scenario:** The Finance team wants to analyze the efficiency of shipments by determining the cost incurred per unit of weight.
* **Action:** Divide `shipment_cost` by `shipment_weight_kg`.
* **Logic:** `shipment_cost / shipment_weight_kg`

**2. Track Shipment Age (`days_since_shipment`)**
* **Scenario:** The Operations team needs to monitor how long it has been since a shipment was dispatched to identify potential delays.
* **Action:** Calculate the difference in days between the `current_date` and the `shipment_date`.
* **Logic:** `datediff(current_date(), shipment_date)`

**3. Compute Tax Liability (`tax_amount`)**
* **Scenario:** For invoicing and compliance, we must calculate the Goods and Services Tax (GST) applicable to each shipment.
* **Action:** Calculate 18% GST on the total `shipment_cost`.
* **Logic:** `shipment_cost * 0.18`


In [0]:
from pyspark.sql.functions import col
rf1 = spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_source1",header = True,inferSchema = True).dropDuplicates()
rf2 = spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_source2",header = True, inferSchema = True).dropDuplicates()
rcsv = rf1.unionByName(rf2, allowMissingColumns=True).where("shipment_id != 'ten'")
#display(rcsv)
#rjson = spark.read.json("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_shipment_detail_3000.json")

jsondf = spark.read \
    .option("mode", "PERMISSIVE") \
    .option("multiLine", "true")\
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .json("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_shipment_detail_3000.json")
#display(jsondf)

jsondf1 = jsondf.withColumn("shipmentid_key",col("shipment_id"))
display(jsondf1)

lfdf = rcsv.join(jsondf1,how = 'left',on='shipment_id').filter("shipmentid_key IS NULL")
display(lfdf)


In [0]:
#1.Calculate Unit Cost (cost_per_kg)
#Scenario: The Finance team wants to analyze the efficiency of shipments by determining the cost incurred per unit of weight.
#Action: Divide shipment_cost by shipment_weight_kg.
#Logic: shipment_cost / shipment_weight_kg

#2. Track Shipment Age (days_since_shipment)
#Scenario: The Operations team needs to monitor how long it has been since a shipment was dispatched to identify potential delays.
#Action: Calculate the difference in days between the current_date and the shipment_date.
#Logic: datediff(current_date(), shipment_date)

#3. Compute Tax Liability (tax_amount)
#Scenario: For invoicing and compliance, we must calculate the Goods and Services Tax (GST) applicable to each shipment.
#Action: Calculate 18% GST on the total shipment_cost.
#Logic: shipment_cost * 0.18

jsondf6 = jsondf5.withColumn("cost_per_kg",round(col("shipment_cost")/col("shipment_weight_kg"),2)).\
withColumn("days_since_shipment",date_diff(current_date(),col("shipment_date"))).\
withColumn("tax_amount",round((col("shipment_cost")*0.18),2))
display(jsondf6)



###### Remove/Eliminate (drop, select, selectExpr)
*Excluding unnecessary or redundant columns to optimize storage and privacy.*<br>
Source File: logistics_source1 and logistics_source2<br>

**1. Remove Redundant Name Columns**
* **Scenario:** Since we have already created the `full_name` column in the Enrichment step, the individual name columns are now redundant and clutter the dataset.
* **Action:** Drop the `first_name` and `last_name` columns.
* **Logic:** `df.drop("first_name", "last_name")`

In [0]:
rawdf15 = rawdf14.drop("staff_first_name","staff_last_name").select("shipment_id","fullname","age","role","hub_location","vehicle_type","data_source","load_dt")
display(rawdf15.limit(20))

##### Splitting & Merging/Melting of Columns
*Reshaping columns to extract hidden values or combine fields for better analysis.*<br>
Source File: logistics_shipment_detail_3000.json<br>
**1. Splitting (Extraction)**
*Breaking one column into multiple to isolate key information.*
* **Split Order Code:**
  * **Action:** Split `order_id` ("ORD100000") into two new columns:
    * `order_prefix` ("ORD")
    * `order_sequence` ("100000")
* **Split Date:**
  * **Action:** Split `shipment_date` into three separate columns for partitioning:
    * `ship_year` (2024)
    * `ship_month` (4)
    * `ship_day` (23)

**2. Merging (Concatenation)**
*Combining multiple columns into a single unique identifier or description.*
* **Create Route ID:**
  * **Action:** Merge `source_city` ("Chennai") and `destination_city` ("Pune") to create a descriptive route key:
    * `route_lane` ("Chennai->Pune")

In [0]:
from  pyspark.sql.functions import substring,regexp_extract,year,month,day
#1.Splitting (Extraction) Breaking one column into multiple to isolate key information.
#Split Order Code:
#Action: Split order_id ("ORD100000") into two new columns:
#order_prefix ("ORD")
#order_sequence ("100000")

#2.Split Date:
#Action: Split shipment_date into three separate columns for partitioning:
#ship_year (2024)
#ship_month (4)
#ship_day (23)

#3.Merging (Concatenation) Combining multiple columns into a single unique identifier or description.
#Create Route ID:
#Action: Merge source_city ("Chennai") and destination_city ("Pune") to create a descriptive route key:
#route_lane ("Chennai->Pune")

#jsondf7 = jsondf6.withColumn("order_prefix",substring("order_id",1,3)).withColumn("order_sequence",substring("order_id",4,6))
jsondf7 = jsondf6.withColumn("order_prefix",regexp_extract(col("order_id"),r"([A-Za-z]+)",1)).\
                  withColumn("order_squence",regexp_extract(col("order_id"),r"([0-9]+)",1)).\
                  withColumn("ship_year",year(col("shipment_date"))).\
                  withColumn("ship_month",month(col("shipment_date"))).\
                  withColumn("ship_day",day(col("shipment_date"))).\
                  withColumn("route_lane",concat(col("source_city"),lit("->"),col("destination_city")))                  
display(jsondf7.limit(20))





## 3. Data Customization & Processing - Application of Tailored Business Specific Rules

### **UDF1: Complex Incentive Calculation**
**Scenario:** The Logistics Head wants to calculate a "Performance Bonus" for drivers based on tenure and role complexity.

**Action:** Create a Python function `calculate_bonus(role, age)` and register it as a Spark UDF.

**Logic:**
* **IF** `Role` == 'Driver' **AND** `Age` > 50:
  * `Bonus` = 15% of Salary (Reward for Seniority)
* **IF** `Role` == 'Driver' **AND** `Age` < 30:
  * `Bonus` = 5% of Salary (Encouragement for Juniors)
* **ELSE**:
  * `Bonus` = 0

**Result:** A new derived column `projected_bonus` is generated for every row in the dataset.

---

### **UDF2: PII Masking (Privacy Compliance)**
**Scenario:** For the analytics dashboard, we must hide the full identity of the staff to comply with privacy laws (GDPR/DPDP), while keeping names recognizable for internal managers.

**Business Rule:** Show the first 2 letters, mask the middle characters with `****`, and show the last letter.

**Action:** Create a UDF `mask_identity(name)`.

**Example:**
* **Input:** `"Rajesh"`
* **Output:** `"Ra****h"`
<br>
**Note: Convert the above udf logic to inbult function based transformation to ensure the performance is improved.**

In [0]:
#1.UDF1: Complex Incentive Calculation
#Scenario: The Logistics Head wants to calculate a "Performance Bonus" for drivers based on tenure and role complexity.
#Action: Create a Python function calculate_bonus(role, age) and register it as a Spark UDF.
#Logic:
#IF Role == 'Driver' AND Age > 50:
#Bonus = 15% of Salary (Reward for Seniority)
#IF Role == 'Driver' AND Age < 30:
#Bonus = 5% of Salary (Encouragement for Juniors)
#ELSE:
#Bonus = 0
#Result: A new derived column projected_bonus is generated for every row in the dataset.

#2.Scenario: For the analytics dashboard, we must hide the full identity of the staff to comply with privacy laws (GDPR/DPDP), while keeping names recognizable for internal managers.
#Business Rule: Show the first 2 letters, mask the middle characters with ****, and show the last letter.
#Action: Create a UDF mask_identity(name).
#Example:
#Input: "Rajesh"
#Output: "Ra****h"
#**Note: Convert the above udf logic to inbult function based transformation to ensure the performance is improved.**

from pyspark.sql.functions import udf

def cal_bonus(role,age):
    if role == 'driver' and age > 50:
        return 0.15
    elif role == 'driver' and age < 30:
        return 0.05
    else:
        return 0  
    
def mask_identity(name):
    return name[0:2]+"****"+name[-1]



In [0]:
bonusfn = udf(cal_bonus)
masknmfn = udf(mask_identity)
rawdf16 = rawdf15.withColumn("projected_bonus",bonusfn(col("role"),col("age"))).withColumn("fullname",masknmfn(col("fullname")))
display(rawdf16)
rawdf17 = rawdf15.withColumn("projected_bonus", 
                             when ((col("role") == 'driver') & (col('age') > 50),0.15).
                             when ((col("role") == 'driver') & (col('age') < 30),0.05).
                             otherwise(0)).withColumn("fullname",concat(substring(col("fullname"),0,2),lit("****"),substring(col("fullname"),-1,1)))
display(rawdf17)

## 4. Data Core Curation & Processing (Pre-Wrangling)
*Applying business logic to focus, filter, and summarize data before final analysis.*

**1. Select (Projection)**<br>
Source Files: logistics_source1 and logistics_source2<br>
* **Scenario:** The Driver App team only needs location data, not sensitive HR info.
* **Action:** Select only `first_name`, `role`, and `hub_location`.

**2. Filter (Selection)**<br>
Source File: json<br>
* **Scenario:** We need a report on active operational problems.
* **Action:** Filter rows where `shipment_status` is **'DELAYED'** or **'RETURNED'**.
* **Scenario:** Insurance audit for senior staff.
* **Action:** Filter rows where `age > 50`.

**3. Derive Flags & Columns (Business Logic)**<br>
Source File: json<br>
* **Scenario:** Identify high-value shipments for security tracking.
* **Action:** Create flag `is_high_value` = **True** if `shipment_cost > 50,000`.
* **Scenario:** Flag weekend operations for overtime calculation.
* **Action:** Create flag `is_weekend` = **True** if day is Saturday or Sunday.

**4. Format (Standardization)**<br>
Source File: json<br>
* **Scenario:** Finance requires readable currency formats.
* **Action:** Format `shipment_cost` to string like **"₹30,695.80"**.
* **Scenario:** Standardize city names for reporting.
* **Action:** Format `source_city` to Uppercase (e.g., "chennai" → **"CHENNAI"**).

**5. Group & Aggregate (Summarization)**<br>
Source Files: logistics_source1 and logistics_source2<br>
* **Scenario:** Regional staffing analysis.
* **Action:** Group by `hub_location` and **Count** the number of staff.
* **Scenario:** Fleet capacity analysis.
* **Action:** Group by `vehicle_type` and **Sum** the `shipment_weight_kg`.

**6. Sorting (Ordering)**<br>
Source File: json<br>
* **Scenario:** Prioritize the most expensive shipments.
* **Action:** Sort by `shipment_cost` in **Descending** order.
* **Scenario:** Organize daily dispatch schedule.
* **Action:** Sort by `shipment_date` (Ascending) then `priority_flag` (Descending).

**7. Limit (Top-N Analysis)**<br>
Source File: json<br>
* **Scenario:** Dashboard snapshot of critical delays.
* **Action:** Filter for 'DELAYED', Sort by Cost, and **Limit to top 10** rows.

In [0]:
from pyspark.sql.functions import format_number,count,sum,col
#1. Select (Projection)
#Source Files: logistics_source1 and logistics_source2
#Scenario: The Driver App team only needs location data, not sensitive HR info.
#Action: Select only first_name, role, and hub_location.
projection = rawdf15.select("fullname","role","hub_location")
#display(projection)

#2. Filter (Selection)
#Source File: json
#Scenario: We need a report on active operational problems.
#Action: Filter rows where shipment_status is 'DELAYED' or 'RETURNED'.
selection = jsondf7.filter((col("shipment_status") == 'DELAYED') | (col("shipment_status") == 'CANCELLED'))
#display(selection)

#Scenario: Insurance audit for senior staff.
#Action: Filter rows where age > 50.
selection1 = rawdf15.select("*").filter(col("age") > 50)
#display(selection1)

#3. Derive Flags & Columns (Business Logic)
#Source File: json
#Scenario: Identify high-value shipments for security tracking.
#Action: Create flag is_high_value = True if shipment_cost > 50,000.
#Scenario: Flag weekend operations for overtime calculation.
#Action: Create flag is_weekend = True if day is Saturday or Sunday.

#4. Format (Standardization)
#Source File: json
#Scenario: Finance requires readable currency formats.
#Action: Format shipment_cost to string like "₹30,695.80".
#Scenario: Standardize city names for reporting.
#Action: Format source_city to Uppercase (e.g., "chennai" → "CHENNAI").
flagdf = jsondf7.withColumn("is_high_value",when(col("shipment_cost")>50000,True).\
                                            otherwise(False)).\
                                            withColumn("is_weekend",when(dayofweek(col("shipment_date")).isin([6,7]),True).\
                                            otherwise(False)).\
                                            withColumn("shipment_cost",concat(lit("₹ "),format_number(col("shipment_cost"),2))).\
                                            withColumn("source_city",upper(col("source_city")))

#display(flagdf)

#5. Group & Aggregate (Summarization)
#Source Files: logistics_source1 and logistics_source2
#Scenario: Regional staffing analysis.
#Action: Group by hub_location and Count the number of staff.
grpby = rawdf15.groupBy("hub_location").agg(count("fullname").alias("staff_count")).na.fill("UNKNOWN")
#display(grpby)
#Scenario: Fleet capacity analysis.
#Action: Group by vehicle_type and Sum the shipment_weight_kg.
grpby1 = flagdf.groupBy("vehicle_type").agg((sum("shipment_weight_kg").alias("Fleet")))
#display(grpby1)
#6. Sorting (Ordering)
#Source File: json
#Scenario: Prioritize the most expensive shipments.
#Action: Sort by shipment_cost in Descending order.
sortdf = flagdf.orderBy(col("shipment_cost"),ascending=False)
#display(sortdf)
#Scenario: Organize daily dispatch schedule.
#Action: Sort by shipment_date (Ascending) then priority_flag (Descending).
sortdf1 = flagdf.orderBy(col("shipment_date"),col("shipment_cost"),ascending=[True,False])
#display(sortdf1)

#7. Limit (Top-N Analysis)
#Source File: json
#Scenario: Dashboard snapshot of critical delays.
#Action: Filter for 'DELAYED', Sort by Cost, and Limit to top 10 rows.

jsondf8 = flagdf.filter(col("shipment_status")=="DELAYED").orderBy("shipment_cost",ascending=[False]).limit(10)
display(jsondf8)
#8. Join (Combining)


## 5. Data Wrangling - Transformation & Analytics
*Combining, modeling, and analyzing data to answer complex business questions.*

### **1. Joins**
Source Files:<br>
Left Side (staff_df):<br> logistics_source1 & logistics_source2<br>
Right Side (shipments_df):<br> logistics_shipment_detail_3000.json<br>
#### **1.1 Frequently Used Simple Joins (Inner, Left)**
* **Inner Join (Performance Analysis):**
  * **Scenario:** We only want to analyze *completed work*. Connect Staff to the Shipments they handled.
  * **Action:** Join `staff_df` and `shipments_df` on `shipment_id`.
  * **Result:** Returns only rows where a staff member is assigned to a valid shipment.
* **Left Join (Idle Resource check):**
  * **Scenario:** Find out which staff members are currently *idle* (not assigned to any shipment).
  * **Action:** Join `staff_df` (Left) with `shipments_df` (Right) on `shipment_id`. Filter where `shipments_df.shipment_id` is NULL.

#### **1.2 Infrequent Simple Joins (Self, Right, Full, Cartesian)**
* **Self Join (Peer Finding):**
  * **Scenario:** Find all pairs of employees working in the same `hub_location`.
  * **Action:** Join `staff_df` to itself on `hub_location`, filtering where `staff_id_A != staff_id_B`.
* **Right Join (Orphan Data Check):**
  * **Scenario:** Identify shipments in the system that have *no valid driver* assigned (Data Integrity Issue).
  * **Action:** Join `staff_df` (Left) with `shipments_df` (Right). Focus on NULLs on the left side.
* **Full Outer Join (Reconciliation):**
  * **Scenario:** A complete audit to find *both* idle drivers AND unassigned shipments in one view.
  * **Action:** Perform a Full Outer Join on `shipment_id`.
* **Cartesian/Cross Join (Capacity Planning):**
  * **Scenario:** Generate a schedule of *every possible* driver assignment to *every* pending shipment to run an optimization algorithm.
  * **Action:** Cross Join `drivers_df` and `pending_shipments_df`.

#### **1.3 Advanced Joins (Semi and Anti)**
* **Left Semi Join (Existence Check):**
  * **Scenario:** "Show me the details of Drivers who have *at least one* shipment." (Standard filtering).
  * **Action:** `staff_df.join(shipments_df, "shipment_id", "left_semi")`.
  * **Benefit:** Performance optimization; it stops scanning the right table once a match is found.
* **Left Anti Join (Negation Check):**
  * **Scenario:** "Show me the details of Drivers who have *never* touched a shipment."
  * **Action:** `staff_df.join(shipments_df, "shipment_id", "left_anti")`.

### **2. Lookup**<br>
Source File: logistics_source1 and logistics_source2 (merged into Staff DF)<br>
* **Scenario:** Validation. Check if the `hub_location` in the staff file exists in the corporate `Master_City_List`.
* **Action:** Compare values against a reference list.

### **3. Lookup & Enrichment**<br>
Source File: logistics_source1 and logistics_source2 (merged into Staff DF)<br>
* **Scenario:** Geo-Tagging.
* **Action:** Lookup `hub_location` ("Pune") in a Master Latitude/Longitude table and enrich the dataset by adding `lat` and `long` columns for map plotting.

### **4. Schema Modeling (Denormalization)**<br>
Source Files: All 3 Files (logistics_source1, logistics_source2, logistics_shipment_detail_3000.json)<br>
* **Scenario:** Creating a "Gold Layer" Table for PowerBI/Tableau.
* **Action:** Flatten the Star Schema. Join `Staff`, `Shipments`, and `Vehicle_Master` into one wide table (`wide_shipment_history`) so analysts don't have to perform joins during reporting.

### **5. Windowing (Ranking & Trends)**<br>
Source Files:<br>
logistics_source2: Provides hub_location (Partition Key).<br>
logistics_shipment_detail_3000.json: Provides shipment_cost (Ordering Key)<br>
* **Scenario:** "Who are the Top 3 Drivers by Cost in *each* Hub?"
* **Action:**
  1. Partition by `hub_location`.
  2. Order by `total_shipment_cost` Descending.
  3. Apply `dense_rank()` and `row_number()
  4. Filter where `rank or row_number <= 3`.

### **6. Analytical Functions (Lead/Lag)**<br>
Source File: <br>
logistics_shipment_detail_3000.json<br>
* **Scenario:** Idle Time Analysis.
* **Action:** For each driver, calculate the days elapsed since their *previous* shipment.

### **7. Set Operations**<br>
Source Files: logistics_source1 and logistics_source2<br>
* **Union:** Combining `Source1` (Legacy) and `Source2` (Modern) into one dataset (Already done in Active Munging).
* **Intersect:** Identifying Staff IDs that appear in *both* Source 1 and Source 2 (Duplicate/Migration Check).
* **Except (Difference):** Identifying Staff IDs present in Source 2 but *missing* from Source 1 (New Hires).

### **8. Grouping & Aggregations (Advanced)**<br>
Source Files:<br>
logistics_source2: Provides hub_location and vehicle_type (Grouping Dimensions).<br>
logistics_shipment_detail_3000.json: Provides shipment_cost (Aggregation Metric).<br>
* **Scenario:** The CFO wants a subtotal report at multiple levels:
  1. Total Cost by Hub.
  2. Total Cost by Hub AND Vehicle Type.
  3. Grand Total.
* **Action:** Use `cube("hub_location", "vehicle_type")` or `rollup()` to generate all these subtotals in a single query.

In [0]:
from pyspark.sql.functions import col
rf1 = spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_source1",header = True,inferSchema = True).dropDuplicates()
rf2 = spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_source2",header = True, inferSchema = True).dropDuplicates()
rcsv = rf1.unionByName(rf2, allowMissingColumns=True).where("shipment_id != 'ten'")
#display(rcsv)
#rjson = spark.read.json("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_shipment_detail_3000.json")

jsondf = spark.read \
    .option("mode", "PERMISSIVE") \
    .option("multiLine", "true")\
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .json("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_shipment_detail_3000.json")
#display(jsondf)

jsondf1 = jsondf.withColumn("shipmentid_key",col("shipment_id"))
display(jsondf1)

lfdf = rcsv.join(jsondf1,how = 'left',on='shipment_id').filter("shipmentid_key isNull()")


In [0]:
#1.1 Frequently Used Simple Joins (Inner, Left)
#Inner Join (Performance Analysis):
#Scenario: We only want to analyze completed work. Connect Staff to the Shipments they handled.
#Action: Join staff_df and shipments_df on shipment_id.
#Result: Returns only rows where a staff member is assigned to a valid shipment.
#display(rawdf17)
#display(jsondf7)
inrdf = rawdf17.join(jsondf7,how = "inner",on="shipment_id")
#display(inrdf)

#Left Join (Idle Resource check):
#Scenario: Find out which staff members are currently idle (not assigned to any shipment).
#Action: Join staff_df (Left) with shipments_df (Right) on shipment_id. Filter where shipments_df.shipment_id is NULL.
jsondf7 = jsondf7.withColumn("shipment_key",col("shipment_id"))
lfdf = rawdf17.join(jsondf7,how="left",on="shipment_id").filter(col("shipment_key").isNull())
display(lfdf)


In [0]:
#Self Join (Peer Finding):
#Scenario: Find all pairs of employees working in the same hub_location.
#Action: Join staff_df to itself on hub_location, filtering where shipment_id_A != shipment_id_B.
a = rawdf17.withColumnRenamed("shipment_id", "shipment_id_A")
b = rawdf17.withColumnRenamed("shipment_id", "shipment_id_B")
c = a.join(b, how = "inner", on = "hub_location").filter(a["shipment_id_A"] != b["shipment_id_B"])
#display(c)

#Right Join (Orphan Data Check):
#Scenario: Identify shipments in the system that have no valid driver assigned (Data Integrity Issue).
#Action: Join staff_df (Left) with shipments_df (Right). Focus on NULLs on the left side.

rawdf17 = rawdf17.withColumn("shipment_ky",col("shipment_id"))
rgtdf = rawdf17.join(jsondf7,how = "right", on = "shipment_id").filter("shipment_ky is null")
display(rgtdf)

#Full Outer Join (Reconciliation):
#Scenario: A complete audit to find both idle drivers AND unassigned shipments in one view.
#Action: Perform a Full Outer Join on shipment_id.
fulldf = rawdf3.join(jsondf,how = "full", on="shipment_id")
#display(fulldf)

#Cartesian/Cross Join (Capacity Planning):
#Scenario: Generate a schedule of every possible driver assignment to every pending shipment to run an optimization algorithm.
#Action: Cross Join drivers_df and pending_shipments_df.
cardf = rawdf3.join(jsondf)
#display(cardf)

In [0]:
#Advanced Joins (Semi and Anti)
#Left Semi Join (Existence Check):
#Scenario: "Show me the details of Drivers who have at least one shipment." (Standard filtering).
#Action: staff_df.join(shipments_df, "shipment_id", "left_semi").
#Benefit: Performance optimization; it stops scanning the right table once a match is found.
semidf = rawdf3.join(jsondf, on ="shipment_id", how = "left_semi")
display(semidf.limit(10))

#Left Anti Join (Negation Check):
#Scenario: "Show me the details of Drivers who have never touched a shipment."
#Action: staff_df.join(shipments_df, "shipment_id", "left_anti").
leftantidf = rawdf3.join(jsondf, on ="shipment_id", how = "left_anti")
display(leftantidf)

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

geo_data = [
    ("AbuDhabi", 24.4539, 54.3773),
    ("Ahmedabad", 23.0225, 72.5714),
    ("Amritsar", 31.6340, 74.8723),
    ("Bangalore", 12.9716, 77.5946),
    ("Birmingham", 52.4862, -1.8904),
    ("Boston", 42.3601, -71.0589),
    ("California", 36.7783, -119.4179),
    ("Chennai", 13.0827, 80.2707),
    ("Chicago", 41.8781, -87.6298),
    ("Coimbatore", 11.0168, 76.9558),
    ("Delhi", 28.7041, 77.1025),
    ("Dubai", 25.2048, 55.2708),
    ("HongKong", 22.3193, 114.1694),
    ("Hyderabad", 17.3850, 78.4867),
    ("Indore", 22.7196, 75.8577),
    ("Jaipur", 26.9124, 75.7873),
    ("Kochi", 9.9312, 76.2673),
    ("London", 51.5074, -0.1278),
    ("Lucknow", 26.8467, 80.9462),
    ("MexicoCity", 19.4326, -99.1332),
    ("Mumbai", 19.0760, 72.8777),
    ("NewYork", 40.7128, -74.0060),
    ("Pune", 18.5204, 73.8567),
    ("Scranton", 41.4089, -75.6624),
    ("Singapore", 1.3521, 103.8198),
    ("Texas", 31.9686, -99.9018),
    ("Tokyo", 35.6762, 139.6503)
]

geodf = spark.createDataFrame(geo_data, ["hub_location", "latitude", "longitude"])
#display(geodf)

#2. Lookup
#Source File: DF of logistics_source1 and logistics_source2 (merged into Staff DF)
#Scenario: Validation. Check if the hub_location in the staff file exists in the corporate Master_City_List.
#Action: Compare values against a reference list.
Master_City_List = rawdf2.select("hub_location").distinct()
#display(Master_City_List)
lookupdf = Master_City_List.join(rawdf3, on = "hub_location", how = "semi")
#display(lookupdf)

#3. Lookup & Enrichment
#Source File: DF of logistics_source1 and logistics_source2 (merged into Staff DF)
#Scenario: Geo-Tagging.
#Action: Lookup hub_location ("Pune") in a Master Latitude/Longitude table and enrich the dataset by adding lat and long columns for map plotting.
lookupdf = rawdf3.join(geodf, on = "hub_location", how = "inner").filter("hub_location = 'Pune'")
#display(lookupdf)

#4. Schema Modeling (Denormalization)
#Source Files: DF of All 3 Files (logistics_source1, logistics_source2, logistics_shipment_detail_3000.json)
#Scenario: Creating a "Gold Layer" Table for PowerBI/Tableau.
#Action: Flatten the Star Schema. Join Staff, Shipments, and Vehicle_Master into one wide table (wide_shipment_history) so analysts don't have to perform joins during reporting.

fltdf = rawdf3.join(jsondf, on = "shipment_id", how = "full").filter("role = 'Driver'")
display(fltdf)


In [0]:
from pyspark.sql.functions import dense_rank,desc,row_number,lead,datediff,to_date,lag
from pyspark.sql.window import Window
#5. Windowing (Ranking & Trends)
#Source Files:
#DF of logistics_source2: Provides hub_location (Partition Key).
#logistics_shipment_detail_3000.json: Provides shipment_cost (Ordering Key)
#Scenario: "Who are the Top 3 Drivers by Cost in each Hub?"
#Action:Partition by hub_location.
# Order by total_shipment_cost Descending.
#Apply dense_rank() and `row_number()
#Filter where rank or row_number <= 3.

winddf = fltdf.filter("role = 'Driver' and hub_location is not null").withColumn("rn",row_number().over(Window.partitionBy("hub_location").orderBy(desc("shipment_cost")))).\
    filter("rn <= 3")

display(winddf)


#6. Analytical Functions (Lead/Lag)
#Source File:
#DF of logistics_shipment_detail_3000.json
#Scenario: Idle Time Analysis.
#Action: For each driver, calculate the days elapsed since their previous shipment.
leadlagdf = fltdf.filter("role = 'Driver'").withColumn("days_since_last_shipment", lag("shipment_date", 1).over(Window.partitionBy("role").orderBy(desc("shipment_date")))).\
withColumn("shipment_date",to_date("shipment_date","dd-MM-yy")).withColumn("days_since_last_shipment",to_date("days_since_last_shipment","dd-MM-yy")).\
withColumn("days_elapsed",datediff("shipment_date","days_since_last_shipment"))
display(leadlagdf)

In [0]:
#7. Set Operations
#Source Files: DF of logistics_source1 and logistics_source2
#Union: Combining Source1 (Legacy) and Source2 (Modern) into one dataset (Already done in Active Munging).
#Intersect: Identifying Staff IDs that appear in both Source 1 and Source 2 (Duplicate/Migration Check).
#Except (Difference): Identifying Staff IDs present in Source 2 but missing from Source 1 (New Hires).
unidf = rawdf1.unionByName(rawdf2, allowMissingColumns=True)
display(unidf)
intersectdf = rawdf1.intersect(rawdf2)
#display(intersectdf)
exceptdf = rawdf1.exceptAll(rawdf2).dropDuplicates()
display(exceptdf)  


In [0]:

#8. Grouping & Aggregations (Advanced)
#Source Files:
#DF of logistics_source2: Provides hub_location and vehicle_type (Grouping Dimensions).
#DF of logistics_shipment_detail_3000.json: Provides shipment_cost (Aggregation Metric).
logis2df = spark.read.csv("/Volumes/workspace/wd36schema/ingestion_volume/source/logistics_source2",header = True, inferSchema = True)
logis2df = logis2df.withColumnRenamed("vehicle_type", "lvehicle_type").filter("shipment_id != 'ten'")
flatdf = logis2df.join(jsondf7,how = "left", on = "shipment_id")
grpdf = flatdf.groupBy("hub_location","lvehicle_type").sum("shipment_cost").orderBy(["hub_location","lvehicle_type"]).alias("Final_shipment_cost)")
#display(grpdf)

#Scenario: The CFO wants a subtotal report at multiple levels:
#Total Cost by Hub.
totdf = flatdf.rollup("hub_location").agg(sum("shipment_cost")).orderBy("hub_location",ascending = [False]).alias("rollup_cost")
#display(totdf)
#Total Cost by Hub AND Vehicle Type.
totdf = flatdf.rollup("hub_location","lvehicle_type").agg(sum("shipment_cost").alias("rollup_cost")).orderBy("hub_location","rollup_cost", ascending = [False,False]).alias("rollup_cost")
display(totdf)
#Grand Total.
totdf = flatdf.agg(sum("shipment_cost"))
#display(totdf)
#Action: Use cube("hub_location", "vehicle_type") or rollup() to generate all these subtotals in a single query.
cbdf = flatdf.cube("hub_location","lvehicle_type").agg(sum("shipment_cost").alias("rollup_cost")).orderBy("hub_location","rollup_cost",ascending = [False,False]).alias("rollup_cost")
display(cbdf)


##6. Data Persistance (LOAD)-> Data Publishing & Consumption<br>

Store the inner joined, lookup and enrichment, Schema Modeling, windowing, analytical functions, set operations, grouping and aggregation data into the delta tables.

##7.Take the copy of the above notebook and try to write the equivalent SQL for which ever applicable.